# Phase 4 v2 — Evaluation, Calibration, Explainability, and Domain Adaptation

This notebook evaluates the final models on the locked test sets: Cleveland held-out test, Hungarian, and Swiss. It also performs DeLong comparison, calibration with Platt/isotonic methods, SHAP explainability, and domain-adaptation ablation.

This notebook displays outputs inline only. The production `04_evaluation.py` runner should save tables and figures.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

COLUMN_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "target",
]
RAW_FEATURES = [c for c in COLUMN_NAMES if c != "target"]

SITE_FILES = {
    "cleveland": {
        "filename": "processed.cleveland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
    },
    "hungarian": {
        "filename": "processed.hungarian.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.hungarian.data",
    },
    "swiss": {
        "filename": "processed.switzerland.data",
        "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.switzerland.data",
    },
}

def ensure_raw_files():
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    paths = {}
    for site, info in SITE_FILES.items():
        path = RAW_DATA_DIR / info["filename"]
        paths[site] = path
        if not path.exists() or path.stat().st_size == 0:
            print(f"Downloading {site}...")
            urlretrieve(info["url"], path)
    return paths

def load_site(path, site):
    df = pd.read_csv(path, header=None, names=COLUMN_NAMES, na_values="?")
    for col in COLUMN_NAMES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["target_original"] = df["target"]
    df["target"] = (df["target"] > 0).astype(int)
    df["site"] = site
    df["original_index"] = df.index
    return df

def load_all_sites():
    paths = ensure_raw_files()
    return {site: load_site(path, site) for site, path in paths.items()}

def split_cleveland_v2(cleveland, random_state=RANDOM_STATE):
    # UCI processed Cleveland has no explicit patient ID. Therefore, rows are treated as independent patients.
    # If a patient_id column becomes available later, replace this with GroupShuffleSplit.
    cleveland = cleveland.copy()

    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=random_state)
    rest_idx, test_idx = next(sss_test.split(cleveland, cleveland["target"]))

    rest = cleveland.iloc[rest_idx].copy()
    test = cleveland.iloc[test_idx].copy()

    # Validation is 15% of total. After removing 15% test, validation is 15/85 of the remaining rows.
    val_fraction_of_rest = 0.15 / 0.85
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_fraction_of_rest, random_state=random_state)
    train_rel_idx, val_rel_idx = next(sss_val.split(rest, rest["target"]))

    train = rest.iloc[train_rel_idx].copy()
    validation = rest.iloc[val_rel_idx].copy()

    train["partition"] = "train"
    validation["partition"] = "validation"
    test["partition"] = "test"

    return train, validation, test

def verify_no_overlap(train, validation, test):
    sets = {
        "train": set(train["original_index"]),
        "validation": set(validation["original_index"]),
        "test": set(test["original_index"]),
    }
    assert sets["train"].isdisjoint(sets["validation"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["validation"].isdisjoint(sets["test"])
    print("No Cleveland original_index overlap across train/validation/test.")

def load_phase1_v2_in_memory():
    sites = load_all_sites()
    train, validation, test = split_cleveland_v2(sites["cleveland"])
    verify_no_overlap(train, validation, test)
    return {
        "cleveland_full": sites["cleveland"],
        "cleveland_train": train,
        "cleveland_validation": validation,
        "cleveland_test": test,
        "hungarian": sites["hungarian"],
        "swiss": sites["swiss"],
    }

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

import optuna
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
XGB_OPTUNA_TRIALS = 50

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "f1": "f1",
    "sensitivity": "recall",
    "specificity": make_scorer(specificity_score),
}
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
class ClinicalFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_df = self._to_dataframe(X)
        self.feature_names_in_ = list(X_df.columns)
        self.feature_names_out_ = list(self.transform(X_df).columns)
        return self

    def transform(self, X):
        X_df = self._to_dataframe(X).copy()
        for col in X_df.columns:
            X_df[col] = pd.to_numeric(X_df[col], errors="coerce")
        X_df["age_x_thalach"] = X_df["age"] * X_df["thalach"]
        X_df["age_x_oldpeak"] = X_df["age"] * X_df["oldpeak"]
        X_df["trestbps_x_chol"] = X_df["trestbps"] * X_df["chol"]
        X_df["cp_x_thal"] = X_df["cp"] * X_df["thal"]
        X_df["exang_x_oldpeak"] = X_df["exang"] * X_df["oldpeak"]
        X_df["framingham_proxy"] = (
            X_df["age"] / 10.0
            + X_df["sex"] * 2.0
            + (X_df["chol"] - 200.0) / 40.0
            + (X_df["trestbps"] - 120.0) / 20.0
            + X_df["fbs"] * 1.5
        )
        return X_df

    def get_feature_names_out(self, input_features=None):
        return np.array(getattr(self, "feature_names_out_", input_features if input_features is not None else []))

    @staticmethod
    def _to_dataframe(X):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=RAW_FEATURES[: X.shape[1]])

def make_preprocessor():
    return Pipeline([
        ("features", ClinicalFeatureEngineer()),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

class CORALAdapter:
    def __init__(self, eps=1e-5):
        self.eps = eps
    def fit(self, X_source, X_target):
        Xs = np.asarray(X_source, dtype=float)
        Xt = np.asarray(X_target, dtype=float)
        self.source_mean_ = Xs.mean(axis=0)
        self.target_mean_ = Xt.mean(axis=0)
        Cs = np.cov(Xs, rowvar=False) + self.eps * np.eye(Xs.shape[1])
        Ct = np.cov(Xt, rowvar=False) + self.eps * np.eye(Xt.shape[1])
        self.source_transform_ = self._mpower(Cs, -0.5)
        self.target_transform_ = self._mpower(Ct, 0.5)
        return self
    def transform_source_to_target(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.source_mean_) @ self.source_transform_ @ self.target_transform_ + self.target_mean_
    @staticmethod
    def _mpower(matrix, power):
        vals, vecs = np.linalg.eigh(matrix)
        vals = np.maximum(vals, 1e-12)
        return vecs @ np.diag(vals ** power) @ vecs.T

def summarize_cv(name, results):
    row = {"model": name}
    for metric in scoring:
        values = results[f"test_{metric}"]
        row[f"{metric}_mean"] = float(np.mean(values))
        row[f"{metric}_std"] = float(np.std(values))
    return row

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
try:
    from sklearn.frozen import FrozenEstimator
except Exception:
    FrozenEstimator = None
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

import shap

RANDOM_STATE = 42

def metric_row(model_name, split_name, y_true, y_prob, threshold=0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "split": split_name,
        "auc_roc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "auc_pr": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bins[i], bins[i+1]
        mask = (y_prob >= left) & (y_prob <= right) if i == n_bins - 1 else (y_prob >= left) & (y_prob < right)
        if np.any(mask):
            ece += abs(y_true[mask].mean() - y_prob[mask].mean()) * mask.mean()
    return float(ece)

def make_calibrated_prefit_model(prefit_model, X_cal, y_cal, method):
    if FrozenEstimator is not None:
        calibrator = CalibratedClassifierCV(estimator=FrozenEstimator(prefit_model), method=method, cv=None)
    else:
        calibrator = CalibratedClassifierCV(estimator=prefit_model, method=method, cv="prefit")
    calibrator.fit(X_cal, y_cal)
    return calibrator

In [ ]:
# DeLong implementation adapted for paired ROC-AUC comparison.
def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2

def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]
    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def delong_roc_test(y_true, pred_one, pred_two):
    y_true = np.asarray(y_true)
    order = np.argsort(-y_true)
    label_1_count = int(np.sum(y_true))
    preds = np.vstack([pred_one, pred_two])[:, order]
    aucs, cov = fast_delong(preds, label_1_count)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return aucs, diff, np.nan, np.nan
    z = abs(diff) / np.sqrt(var)
    p = 2 * (1 - norm.cdf(abs(z)))
    return aucs, diff, z, p

## Load splits and train final models for notebook evaluation

The production runner should load saved Phase 3 models. This notebook can rebuild them from saved parameters if available, or use the best parameters printed from Phase 3.

In [ ]:
phase1 = load_phase1_v2_in_memory()
train = phase1["cleveland_train"].copy()
validation = phase1["cleveland_validation"].copy()
cleveland_test = phase1["cleveland_test"].copy()
hungarian = phase1["hungarian"].copy()
swiss = phase1["swiss"].copy()

X_train = train[RAW_FEATURES]
y_train = train["target"]
X_val = validation[RAW_FEATURES]
y_val = validation["target"]
X_dev = pd.concat([X_train, X_val], axis=0)
y_dev = pd.concat([y_train, y_val], axis=0)

test_sets = {
    "cleveland_test": (cleveland_test[RAW_FEATURES], cleveland_test["target"]),
    "hungarian": (hungarian[RAW_FEATURES], hungarian["target"]),
    "swiss": (swiss[RAW_FEATURES], swiss["target"]),
}

for name, (X_split, y_split) in test_sets.items():
    print(f"{name:15s}: X={X_split.shape}, positive_rate={y_split.mean():.2%}")

In [ ]:
# Best parameters from the current Phase 3 run can be overridden by reading phase3_best_params.json in the runner.
lr_params = {"C": 0.5, "class_weight": None, "penalty": "l1", "solver": "liblinear"}
xgb_params = {'n_estimators': 93, 'max_depth': 4, 'learning_rate': 0.12429497044775645, 'subsample': 0.709474251299767, 'colsample_bytree': 0.6546222785562367, 'min_child_weight': 6.077498591972471, 'gamma': 2.4113098256079746, 'reg_alpha': 0.032011336793843194, 'reg_lambda': 0.6231946242788892, 'scale_pos_weight': 0.7972900365412258}
mlp_params = {"solver": "adam", "learning_rate_init": 0.01, "hidden_layer_sizes": (32,), "batch_size": 16, "alpha": 0.1, "activation": "tanh"}

models = {
    "Logistic Regression": Pipeline([("preprocess", make_preprocessor()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE, **lr_params))]),
    "XGBoost + Optuna": Pipeline([("preprocess", make_preprocessor()), ("model", XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist", **xgb_params))]),
    "Small MLP": Pipeline([("preprocess", make_preprocessor()), ("model", WeightedMLPClassifier(max_iter=1200, early_stopping=True, validation_fraction=0.20, n_iter_no_change=40, random_state=RANDOM_STATE, **mlp_params))]),
}

for model in models.values():
    model.fit(X_dev, y_dev)
print("Final notebook models fitted on Cleveland train + validation only.")

## Performance on locked test sets

In [ ]:
rows = []
predictions = {}
for model_name, model in models.items():
    for split_name, (X_split, y_split) in test_sets.items():
        prob = model.predict_proba(X_split)[:, 1]
        predictions[(model_name, split_name)] = prob
        rows.append(metric_row(model_name, split_name, y_split, prob))
performance_summary = pd.DataFrame(rows).sort_values(["split", "auc_roc"], ascending=[True, False])
performance_summary

## DeLong test between top two models

The test is run separately per test split using paired predictions from the top two AUC models in that split.

In [ ]:
delong_rows = []
for split_name, (X_split, y_split) in test_sets.items():
    top_two = performance_summary[performance_summary["split"] == split_name].sort_values("auc_roc", ascending=False).head(2)["model"].tolist()
    if len(top_two) < 2 or len(np.unique(y_split)) < 2:
        continue
    aucs, diff, z, p = delong_roc_test(y_split, predictions[(top_two[0], split_name)], predictions[(top_two[1], split_name)])
    delong_rows.append({"split": split_name, "model_1": top_two[0], "model_2": top_two[1], "auc_1": aucs[0], "auc_2": aucs[1], "auc_diff": diff, "z": z, "p_value": p})
delong_summary = pd.DataFrame(delong_rows)
delong_summary

## Calibration: Platt scaling and isotonic regression

Base models are trained on Cleveland train. Calibration is fitted only on Cleveland validation, then evaluated on the locked test sets.

In [ ]:
calibrated_models = {}
for model_name, model in models.items():
    base = clone(model)
    base.fit(X_train, y_train)
    calibrated_models[(model_name, "uncalibrated")] = base
    calibrated_models[(model_name, "platt_sigmoid")] = make_calibrated_prefit_model(base, X_val, y_val, method="sigmoid")
    calibrated_models[(model_name, "isotonic")] = make_calibrated_prefit_model(base, X_val, y_val, method="isotonic")

cal_rows = []
for (model_name, calibration), model in calibrated_models.items():
    for split_name, (X_split, y_split) in test_sets.items():
        prob = model.predict_proba(X_split)[:, 1]
        cal_rows.append({"model": model_name, "calibration": calibration, "split": split_name, "ece": expected_calibration_error(y_split, prob), "auc_roc": roc_auc_score(y_split, prob) if len(np.unique(y_split)) == 2 else np.nan})
calibration_summary = pd.DataFrame(cal_rows).sort_values(["split", "model", "ece"])
calibration_summary

In [ ]:
for split_name, (X_split, y_split) in test_sets.items():
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot([0, 1], [0, 1], linestyle="--")
    for (model_name, calibration), model in calibrated_models.items():
        if model_name != performance_summary[performance_summary["split"] == split_name].sort_values("auc_roc", ascending=False).iloc[0]["model"]:
            continue
        prob = model.predict_proba(X_split)[:, 1]
        frac_pos, mean_pred = calibration_curve(y_split, prob, n_bins=10, strategy="uniform")
        ax.plot(mean_pred, frac_pos, marker="o", label=calibration)
    ax.set_title(f"Reliability diagram: {split_name}")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed positive fraction")
    ax.legend()
    plt.tight_layout()
    plt.show()

## SHAP explainability

TreeExplainer is used for XGBoost because it is the tree-based model required for SHAP analysis. If another model has the best AUC, XGBoost is still used as the SHAP-compatible model for this professor requirement.

In [ ]:
xgb_model = models["XGBoost + Optuna"]
pre = xgb_model.named_steps["preprocess"]
booster = xgb_model.named_steps["model"]
X_dev_transformed = pre.transform(X_dev)
feature_names = pre.named_steps["features"].get_feature_names_out()

explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_dev_transformed)
shap.summary_plot(shap_values, X_dev_transformed, feature_names=feature_names, show=True)

importance = pd.DataFrame({"feature": feature_names, "mean_abs_shap": np.abs(shap_values).mean(axis=0)}).sort_values("mean_abs_shap", ascending=False)
importance.head(15)

## Domain adaptation ablation

This notebook uses CORAL as an isolated ablation. It adapts processed target features before applying each model's final classifier. The test labels are used only for evaluation after predictions are generated.

In [ ]:
def predict_with_coral(model, X_source_dev, X_target):
    pre = model.named_steps["preprocess"]
    clf = model.named_steps["model"]
    Xs = pre.transform(X_source_dev)
    Xt = pre.transform(X_target)
    adapter = CORALAdapter().fit(Xs, Xt)
    Xt_aligned = adapter.transform_source_to_target(Xt)
    return clf.predict_proba(Xt_aligned)[:, 1]

ablation_rows = []
for model_name, model in models.items():
    for split_name, (X_split, y_split) in test_sets.items():
        base_prob = model.predict_proba(X_split)[:, 1]
        coral_prob = predict_with_coral(model, X_dev, X_split)
        base = metric_row(model_name, split_name, y_split, base_prob); base["domain_adaptation"] = "none"
        coral = metric_row(model_name, split_name, y_split, coral_prob); coral["domain_adaptation"] = "coral"
        ablation_rows.extend([base, coral])

domain_adaptation_ablation = pd.DataFrame(ablation_rows)
domain_adaptation_ablation.sort_values(["split", "model", "domain_adaptation"])

In [ ]:
delta_rows = []
for (model_name, split_name), g in domain_adaptation_ablation.groupby(["model", "split"]):
    if set(g["domain_adaptation"]) >= {"none", "coral"}:
        none = g[g["domain_adaptation"] == "none"].iloc[0]
        coral = g[g["domain_adaptation"] == "coral"].iloc[0]
        delta_rows.append({
            "model": model_name,
            "split": split_name,
            "auc_roc_delta_coral_minus_none": coral["auc_roc"] - none["auc_roc"],
            "f1_delta_coral_minus_none": coral["f1"] - none["f1"],
            "sensitivity_delta_coral_minus_none": coral["sensitivity"] - none["sensitivity"],
            "specificity_delta_coral_minus_none": coral["specificity"] - none["specificity"],
        })
coral_delta_summary = pd.DataFrame(delta_rows)
coral_delta_summary

## Phase 4 conclusion checklist

After running this notebook, document: best model per test split, DeLong p-values, calibration method with lowest ECE, SHAP top features, and whether CORAL improves or hurts each external split. These concrete numbers will drive Phase 5.